### Import Libraries

In [1]:
import pandas as pd 
import numpy as np
from datetime import date 

### Load Files

In [2]:
try:
    opportunities = pd.read_csv('opportunities.csv', parse_dates=['engage_date', 'close_date'])
    accounts = pd.read_csv('accounts.csv', usecols=['account', 'office_location'])
    products = pd.read_csv('products.csv', usecols=['product', 'series'])
    sales_team = pd.read_csv('sales_teams.csv', usecols=['sales_agent', 'regional_office'])
except FileNotFoundError:
    print("Error: One or more required CSV files were not found.")
    exit()


for name, df in [
    ('opportunities', opportunities), 
    ('accounts', accounts), 
    ('products', products), 
    ('sales_team', sales_team)]:
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    print(df.columns.tolist())
    print()

opportunities: 14343 rows x 12 columns
['opportunity_id', 'opportunity_name', 'sales_agent', 'product', 'account', 'lead_id', 'deal_stage', 'type', 'engage_date', 'close_date', 'close_value', 'renewal_source']

accounts: 62 rows x 2 columns
['account', 'office_location']

products: 12 rows x 2 columns
['product', 'series']

sales_team: 35 rows x 2 columns
['sales_agent', 'regional_office']



### Apply filters and merge datasets for won deals only

In [3]:
# Filter won deals
df_won = opportunities[opportunities['deal_stage'] == 'Won']

# Data quality checks
df_won = df_won.dropna(subset=['close_value'])
df_won = df_won[df_won['close_value'] > 0]
df_won['close_date'] = pd.to_datetime(df_won['close_date'])
df_won['engage_date'] = pd.to_datetime(df_won['engage_date'])

# Merge with accounts to get office location
df_won = df_won.merge(accounts, on='account', how='left')
# Merge with products to get series
df_won = df_won.merge(products, on='product', how='left')

# Days to close calculation
df_won['days_to_close'] = (df_won['close_date'] - df_won['engage_date']).dt.days

# Checkpoint of won deals after merging
print(f"Won deals after merging: {df_won.shape[0]} rows")

Won deals after merging: 4717 rows


In [4]:
# Define quarter end months and cutoff day
QUARTER_END_MONTHS = [3, 6, 9, 12]
QUARTER_END_CUTOFF_DAY = 15

def is_quarter_end(date):
    return int(date.month in QUARTER_END_MONTHS and date.day >= QUARTER_END_CUTOFF_DAY)

### Create training set

In [5]:
# Define features to group by
features = [
    "close_date",
    "product",
    "series",
    "office_location",
    "type"
]

# Aggregate data to create training set
training_df = (
    df_won
    .groupby(features, as_index=False)
    .agg(
        total_revenue=("close_value", "sum"),
        total_deals=("opportunity_id", "count"),
        avg_days_to_close=("days_to_close", "mean")
    )
)

# Create average deal size feature
training_df["avg_deal_size"] = (
    training_df["total_revenue"] / training_df["total_deals"]
)

# Extract date features
training_df["month"] = training_df["close_date"].dt.month
training_df["day_of_week"] = training_df["close_date"].dt.dayofweek
training_df["quarter"] = training_df["close_date"].dt.quarter
training_df["year"] = training_df["close_date"].dt.year

# Create quarter end flag
training_df["quarter_end_flag"] = training_df["close_date"].apply(is_quarter_end)

# Final column order
final_cols = [
    "close_date",
    "month",
    "day_of_week",
    "quarter",
    "year",
    "product",
    "series",
    "office_location",
    "type",
    "total_revenue",
    "total_deals",
    "avg_deal_size",
    "avg_days_to_close",
    "quarter_end_flag"
]

# Reorder, sort, reset index
training_df = (
    training_df[final_cols]
    .sort_values("close_date")
    .reset_index(drop=True)
)

In [6]:
print("Final training_df shape:", training_df.shape)

print("\nDate range:")
print("Min close_date:", training_df["close_date"].min())
print("Max close_date:", training_df["close_date"].max())

print("\nYears present:")
print(sorted(training_df["year"].unique()))

print("\nNull counts:")
print(training_df.isnull().sum())

training_total_revenue = training_df["total_revenue"].sum()
won_total_close_value = df_won["close_value"].sum()

print("\nRevenue check:")
print("training_df total_revenue:", training_total_revenue)
print("df_won close_value total:", won_total_close_value)
print("Difference:", training_total_revenue - won_total_close_value)

if np.isclose(training_total_revenue, won_total_close_value):
    print("Revenue totals match.")
else:
    print("Revenue totals do NOT match — investigate filters or aggregation.")

print("\nPreview:")
display(training_df.head())

Final training_df shape: (4659, 14)

Date range:
Min close_date: 2021-01-25 00:00:00
Max close_date: 2024-12-30 00:00:00

Years present:
[np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

Null counts:
close_date           0
month                0
day_of_week          0
quarter              0
year                 0
product              0
series               0
office_location      0
type                 0
total_revenue        0
total_deals          0
avg_deal_size        0
avg_days_to_close    0
quarter_end_flag     0
dtype: int64

Revenue check:
training_df total_revenue: 20490721.810000002
df_won close_value total: 20490721.810000002
Difference: 0.0
Revenue totals match.

Preview:


,close_date,month,day_of_week,quarter,year,product,series,office_location,type,total_revenue,total_deals,avg_deal_size,avg_days_to_close,quarter_end_flag
0,2021-01-25,1,0,1,2021,Analytics Basic,Analytics,United Kingdom,Upsell,2343.07,1,2343.07,21.0,0
1,2021-01-27,1,2,1,2021,DataSync Connector,Add-Ons,Japan,Renewal,2844.79,1,2844.79,19.0,0
2,2021-01-28,1,3,1,2021,Integration Pack,Add-Ons,Japan,New Business,1479.50,1,1479.50,22.0,0
3,2021-02-03,2,2,1,2021,Integration Pack,Add-Ons,Singapore,Renewal,1352.25,1,1352.25,22.0,0
4,2021-02-10,2,2,1,2021,SupportDesk Pro,SupportDesk,UAE,Renewal,2979.73,1,2979.73,36.0,0


In [7]:
# Save
training_df.to_csv("train_daily_revenue.csv", index=False)
print("Saved train_daily_revenue.csv")

Saved train_daily_revenue.csv


### Building Prediction Template

In [8]:
# We use the top 50 most active product × office_location × type combinations
# from the training data rather than a full cross join of all possible segments.
# This avoids forecasting segments with too little historical data for the model
# to learn meaningful patterns from — keeping predictions reliable and noise low.

# Define future date range (full year 2025)
prediction_dates = pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")

# Load training data and identify top 50 most active segments combinations
training_df = pd.read_csv("train_daily_revenue.csv", parse_dates=["close_date"])

top_segments = (
    training_df
    .groupby(["product", "office_location", "type"])
    .size()
    .reset_index(name="count")
    .nlargest(50, "count")
    [['product', 'office_location', 'type']]
)

# Iterate over top segments to avoid a full cartesian product 
predictions_list = []
for _, row in top_segments.iterrows():
    temp_df = pd.DataFrame(prediction_dates, columns=['close_date'])
    temp_df['product']          = row['product']
    temp_df['office_location']  = row['office_location']
    temp_df['type']             = row['type']
    predictions_list.append(temp_df)

predictions_df = pd.concat(predictions_list, ignore_index=True)

# Join product series from training data 
series_mapping = training_df[['product', 'series']].drop_duplicates()
predictions_df = predictions_df.merge(series_mapping, on='product', how='left')

# Extract date features
predictions_df['month']       = predictions_df['close_date'].dt.month
predictions_df['day_of_week'] = predictions_df['close_date'].dt.dayofweek
predictions_df['quarter']     = predictions_df['close_date'].dt.quarter
predictions_df['year']        = predictions_df['close_date'].dt.year

# Derive quarter end flag
predictions_df['quarter_end_flag'] = predictions_df['close_date'].apply(is_quarter_end)

# Add null placeholders for target and other features to be predicted
predictions_df['total_revenue']      = np.nan
predictions_df['total_deals']        = np.nan
predictions_df['avg_deal_size']      = np.nan
predictions_df['avg_days_to_close']  = np.nan

final_cols = [
    'close_date',
    'month',
    'day_of_week',
    'quarter',
    'year',
    'product',
    'series',
    'office_location',
    'type',
    'total_revenue',
    'total_deals',
    'avg_deal_size',
    'avg_days_to_close',
    'quarter_end_flag'
]

predictions_df = (
    predictions_df[final_cols]
    .sort_values('close_date')
    .reset_index(drop=True)
)

print(f"Prediction dataset shape: {predictions_df.shape}")
print(f"Date range: {predictions_df['close_date'].min()} - {predictions_df['close_date'].max()}")
print(f"Segments included: {predictions_df[['product','office_location','type']].drop_duplicates().shape[0]}")
print(f"\nNull counts:\n{predictions_df.isnull().sum()}")
print(f"\nPreview:")
display(predictions_df.head())

predictions_df.to_csv('test_future_dates.csv', index=False)
print("\nSaved test_future_dates.csv")

Prediction dataset shape: (18250, 14)
Date range: 2025-01-01 00:00:00 - 2025-12-31 00:00:00
Segments included: 50

Null counts:
close_date               0
month                    0
day_of_week              0
quarter                  0
year                     0
product                  0
series                   0
office_location          0
type                     0
total_revenue        18250
total_deals          18250
avg_deal_size        18250
avg_days_to_close    18250
quarter_end_flag         0
dtype: int64

Preview:


,close_date,month,day_of_week,quarter,year,product,series,office_location,type,total_revenue,total_deals,avg_deal_size,avg_days_to_close,quarter_end_flag
0,2025-01-01,1,2,1,2025,MarketingHub Lite,MarketingHub,Singapore,New Business,NaN,NaN,NaN,NaN,0
1,2025-01-01,1,2,1,2025,CoreCRM Pro,CoreCRM,Singapore,Renewal,NaN,NaN,NaN,NaN,0
2,2025-01-01,1,2,1,2025,CoreCRM Starter,CoreCRM,Australia,New Business,NaN,NaN,NaN,NaN,0
3,2025-01-01,1,2,1,2025,CoreCRM Starter,CoreCRM,Canada,New Business,NaN,NaN,NaN,NaN,0
4,2025-01-01,1,2,1,2025,CoreCRM Starter,CoreCRM,Japan,Renewal,NaN,NaN,NaN,NaN,0



Saved test_future_dates.csv
